<a href="https://colab.research.google.com/github/simbak-develop/Optimizacion-UACJ/blob/02-Los-tres-ejercios-de-la-presentacion-del-profesor/02-Los_tres_ejercios_de_la_presentacion_del_profesor/06-minimizar_costo_de_transporte/06-minimizar_costo_de_transporte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **INSTALAR LIBRERIA GLPK**

In [ ]:
!apt-get install -y -qq glpk-utils

# **CODIGO FUENTE**

In [1]:
import pyomo.environ as pyo

# Crear el modelo
model = pyo.ConcreteModel()

# Definir un conjunto de índices para las variables
model.vars = pyo.Set(initialize=['f1', 'f2', 'd1', 'd2'])

# Variables de decisión
model.f1 = pyo.Var(within=pyo.NonNegativeIntegers) #Fabrica 1
model.f2 = pyo.Var(within=pyo.NonNegativeIntegers) #Fabrica 2
model.d1 = pyo.Var(within=pyo.NonNegativeIntegers) #Destino 1
model.d2 = pyo.Var(within=pyo.NonNegativeIntegers) #Destino 2

# Función objetivo
model.obj = pyo.Objective(
    expr=4 * model.f1 + 6 * model.f2 + 5 * model.d1 + 4 * model.d2, sense=pyo.minimize
)  # Minimizar el costo

# Restricciones
model.con1 = pyo.Constraint(expr=model.f1 + model.f2 <= 50) #Capacidad fabrica 1
model.con2 = pyo.Constraint(expr=model.d1 + model.d2 <= 60) #Capacidad fabrica 2
model.con3 = pyo.Constraint(expr=model.f1 + model.d1 >= 40) #Demanda destino 1
model.con4 = pyo.Constraint(expr=model.f2 + model.d2 >= 70) #Demanda destino 2

# Función para la restricción de no negatividad
def non_negativity_rule(model, i):
    return model.x[i] >= 0

# Aplicar la restricción de no negatividad a todas las variables en un solo Constraint
model.non_negativity = pyo.Constraint(model.vars, rule=non_negativity_rule)

# Resolver
solver = pyo.SolverFactory('glpk')
result = solver.solve(model)

# Mostrar resultados
print(f"Costo mínimo: {model.obj()}")
print(f"f1 = {model.f1.value}, f2 = {model.f2.value}")
print(f"d1 = {model.d1.value}, d2 = {model.d2.value}")

ERROR: Rule failed when generating expression for Constraint non_negativity
with index f1: AttributeError: 'ConcreteModel' object has no attribute 'x'
ERROR: Constructing component 'non_negativity' from data=None failed:
        AttributeError: 'ConcreteModel' object has no attribute 'x'


AttributeError: 'ConcreteModel' object has no attribute 'x'

# **CODIGO FUENTE VERSION 2**

In [3]:
import pyomo.environ as pyo

# Crear el modelo
model = pyo.ConcreteModel()

# Definir conjuntos de fábricas y destinos
model.fabricas = pyo.Set(initialize=['F1', 'F2'])
model.destinos = pyo.Set(initialize=['D1', 'D2'])

# Variables de decisión: cantidad transportada desde fábrica i a destino j
model.x = pyo.Var(model.fabricas, model.destinos, within=pyo.NonNegativeIntegers)

# Función objetivo: minimizar el costo total de transporte
model.obj = pyo.Objective(
    expr=4 * model.x['F1', 'D1'] + 6 * model.x['F1', 'D2'] + 5 * model.x['F2', 'D1'] + 4 * model.x['F2', 'D2'],
    sense=pyo.minimize
)

# Restricciones de capacidad (oferta)
def capacidad_f1_rule(model):
    return model.x['F1', 'D1'] + model.x['F1', 'D2'] <= 50
model.capacidad_f1 = pyo.Constraint(rule=capacidad_f1_rule)

def capacidad_f2_rule(model):
    return model.x['F2', 'D1'] + model.x['F2', 'D2'] <= 60
model.capacidad_f2 = pyo.Constraint(rule=capacidad_f2_rule)

# Restricciones de demanda
def demanda_d1_rule(model):
    return model.x['F1', 'D1'] + model.x['F2', 'D1'] >= 40
model.demanda_d1 = pyo.Constraint(rule=demanda_d1_rule)

def demanda_d2_rule(model):
    return model.x['F1', 'D2'] + model.x['F2', 'D2'] >= 70
model.demanda_d2 = pyo.Constraint(rule=demanda_d2_rule)

# Resolver
solver = pyo.SolverFactory('glpk')
result = solver.solve(model)

# Mostrar resultados
print(f"Costo mínimo: {model.obj()}")
print("Cantidades transportadas:")
for i in model.fabricas:
    for j in model.destinos:
        print(f"x[{i}, {j}] = {model.x[i, j].value}")

Costo mínimo: 460.0
Cantidades transportadas:
x[F1, D1] = 40.0
x[F1, D2] = 10.0
x[F2, D1] = 0.0
x[F2, D2] = 60.0


# Problema de Transporte: Minimización de Costos

Una empresa necesita transportar productos desde dos fábricas (F1 y F2) a dos destinos (D1 y D2). El objetivo es minimizar el costo total de transporte.

## 1. Declaración del Problema

Las capacidades de las fábricas y las demandas de los destinos son las siguientes:

* **Fábrica 1 (F1):** Capacidad de 50 unidades.
* **Fábrica 2 (F2):** Capacidad de 60 unidades.
* **Destino 1 (D1):** Demanda de 40 unidades.
* **Destino 2 (D2):** Demanda de 70 unidades.

Los costos de transporte por unidad entre cada fábrica y destino son:

* **De F1 a D1:** $4
* **De F1 a D2:** $6
* **De F2 a D1:** $5
* **De F2 a D2:** $4

## 2. Variables de Decisión

Definiremos las siguientes variables de decisión, representando la cantidad de unidades transportadas de cada fábrica a cada destino:

* $x_{11}$: Unidades transportadas de la Fábrica 1 (F1) al Destino 1 (D1)
* $x_{12}$: Unidades transportadas de la Fábrica 1 (F1) al Destino 2 (D2)
* $x_{21}$: Unidades transportadas de la Fábrica 2 (F2) al Destino 1 (D1)
* $x_{22}$: Unidades transportadas de la Fábrica 2 (F2) al Destino 2 (D2)

Todas las variables $x_{ij}$ deben ser no negativas, es decir, $x_{ij} \ge 0$.

## 3. Función Objetivo

El objetivo es minimizar el costo total de transporte. Esto se logra multiplicando el número de unidades transportadas por su costo unitario correspondiente y sumando todos estos productos.

$$
\text{Minimizar Costo Total} = 4x_{11} + 6x_{12} + 5x_{21} + 4x_{22}
$$

## 4. Restricciones

Las restricciones se basan en la capacidad de las fábricas y la demanda de los destinos.

### Restricciones de Capacidad (Suministro):

La cantidad total de unidades que salen de cada fábrica no puede exceder su capacidad.

* **Capacidad de F1:**
    $$
    x_{11} + x_{12} \le 50
    $$

* **Capacidad de F2:**
    $$
    x_{21} + x_{22} \le 60
    $$

### Restricciones de Demanda:

La cantidad total de unidades que llegan a cada destino debe satisfacer su demanda.

* **Demanda de D1:**
    $$
    x_{11} + x_{21} \ge 40
    $$

* **Demanda de D2:**
    $$
    x_{12} + x_{22} \ge 70
    $$

    * **No valores menor a zero:**
    $$
     x_{11} + x_{12} + x_{21} + x_{22} \ge 70
    $$


